# A1 Pairs/OU — 01 · Data Prep

**Run this on the VPS** (the 20-level depth lives at `~/data/tbt-dhan/depth`, and the
collector venv has `pyarrow`):

```
cd ~/paper-trader && ~/collector-dhan/venv/bin/jupyter nbconvert --to notebook \
    --execute research/pairs_ou/01_data_prep.ipynb
```
or just run the cells in an SSH Jupyter session.

It builds **minute mid-bars** for the universe across *all available* trading days,
plus a per-symbol cost table (median spread, price, lot), and saves a compact
`panel.parquet` + `symbol_stats.csv` you can sync down and feed to NB2/NB3.

> Honest note up front: cointegration wants a long, varied span. The core 3
> (HDFCBANK/ICICIBANK/RELIANCE) have the most history; the new names start ~mid-June.
> Treat short-history pairs' results as *directional*, not a verdict.

In [ ]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd

# ── config ────────────────────────────────────────────────────────────────────
DATA_ROOT = os.path.expanduser("~/data/tbt-dhan/depth")   # VPS depth root
SYMBOLS   = ["HDFCBANK", "ICICIBANK", "RELIANCE", "SBIN", "AXISBANK", "BHARTIARTL", "ITC"]
BAR_FREQ  = "1min"                                          # mid-bar frequency
OUT_DIR   = Path(os.path.expanduser("~/paper-trader/research/pairs_ou/out"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOT_SIZES = {"HDFCBANK":550,"ICICIBANK":700,"RELIANCE":500,
             "SBIN":750,"AXISBANK":625,"BHARTIARTL":475,"ITC":1600}
print("depth root:", DATA_ROOT, "| exists:", os.path.isdir(DATA_ROOT))

In [ ]:
# ── loader: clean L1 mid + spread for one symbol-day (mirrors basecamp_recon.markout) ──
L1 = ["collector_received_at", "bid_price_01", "ask_price_01"]

def symbol_day_files(name, date_dir):
    hits = sorted(glob.glob(f"{date_dir}/symbol={name}-*/compacted-*.parquet"))
    if not hits:
        hits = sorted(glob.glob(f"{date_dir}/symbol={name}-*/*.parquet"))
    return hits

def load_symbol_day(name, date_dir):
    files = symbol_day_files(name, date_dir)
    if not files:
        return None
    comp = [p for p in files if "compacted" in Path(p).name]
    files = comp if comp else files
    df = pd.concat([pd.read_parquet(p, columns=L1) for p in files], ignore_index=True)
    bp, ap = df.bid_price_01.to_numpy(float), df.ask_price_01.to_numpy(float)
    good = (bp > 0) & (ap > 0) & (ap >= bp)
    df = df.loc[good]
    if df.empty:
        return None
    ts = pd.to_datetime(df.collector_received_at, utc=True)
    bp, ap = df.bid_price_01.to_numpy(float), df.ask_price_01.to_numpy(float)
    out = pd.DataFrame({"ts": ts.to_numpy(), "mid": (bp+ap)/2.0, "spread": ap-bp})
    return out.sort_values("ts")

def minute_bars(name):
    """Concatenate minute mid-bars for `name` across all available trading days."""
    parts, spreads = [], []
    for date_dir in sorted(glob.glob(f"{DATA_ROOT}/trading_date=*")):
        d = load_symbol_day(name, date_dir)
        if d is None:
            continue
        d = d.set_index("ts")
        bars = d["mid"].resample(BAR_FREQ).last().dropna()
        parts.append(bars)
        spreads.append(d["spread"].median())
    if not parts:
        return None, np.nan
    series = pd.concat(parts).sort_index()
    series = series[~series.index.duplicated(keep="last")]
    return series, float(np.nanmedian(spreads))

In [ ]:
# ── build the panel + coverage report ──────────────────────────────────────────
cols, stats = {}, []
for s in SYMBOLS:
    ser, med_spread = minute_bars(s)
    if ser is None or ser.empty:
        print(f"{s:11} NO DATA"); continue
    cols[s] = ser
    stats.append({"symbol": s, "n_bars": len(ser),
                  "first": ser.index.min(), "last": ser.index.max(),
                  "n_days": ser.index.normalize().nunique(),
                  "med_price": round(float(ser.median()),2),
                  "med_spread": round(med_spread,4),
                  "lot": LOT_SIZES.get(s)})
panel = pd.DataFrame(cols).sort_index()
stats = pd.DataFrame(stats).set_index("symbol")
print("panel shape:", panel.shape)
stats

In [ ]:
# ── coverage heat: bars per symbol per day ──────────────────────────────────────
import matplotlib.pyplot as plt
daily = panel.notna().groupby(panel.index.normalize()).sum()
print("days x symbols:", daily.shape)
ax = (panel / panel.bfill().iloc[0]).plot(figsize=(13,5), title="normalized mid (sanity)")
ax.legend(loc="upper left", ncol=4, fontsize=8); plt.show()
daily.tail(15)

In [ ]:
# ── save compact artifacts to sync down for NB2/NB3 ─────────────────────────────
panel.to_parquet(OUT_DIR / "panel.parquet")
stats.to_csv(OUT_DIR / "symbol_stats.csv")
print("wrote:", OUT_DIR / "panel.parquet", "and symbol_stats.csv")
print("\nSync down with:")
print(f"  rsync -avz lightsail-mumbai:{OUT_DIR}/ ./research/pairs_ou/out/")

**Next:** sync `out/` to your laptop and run `02_cointegration_screen.ipynb` (it needs
`statsmodels`, which is local-only). NB1 is the only notebook that must run on the VPS.